- _exponent._bronze_allscripts_tw_works.dbo_encounter_other
- _exponent._bronze_allscripts_tw_works.dbo_encounter_itemchild
- _exponent._bronze_allscripts_tw_works.dbo_site_de

### Source Tables:
- _exponent._bronze_allscripts_tw_works_vw.dbo_visit (main visit table)
- _exponent._bronze_allscripts_tw_works_vw.dbo_visit_detail (admission/discharge details)
- _exponent._bronze_allscripts_tw_works_vw.dbo_visit_type_de (visit type lookup)

### To Do:
- Map VisitTypeDE to OMOP visit_concept_id using domain_source_to_concept
- Map visit_type_concept_id (32817 = EHR is default)
- Link to provider_id using PrimaryProviderID once provider table is populated
- Link to care_site_id using LocationDE once care_site table is populated
- Map admitted_from and discharged_to concepts

### Notes:
- PERSON must run before VISIT_OCCURRENCE (needs person_id mapping)
- PROVIDER and CARE_SITE should ideally run first but are optional
- dbo_visit.ID is the visit identifier (229M unique visits)
- dbo_visit.PatientID links to dbo_person.ID
- Visit types: 2=Office, 3=Telephone, 4=Inpatient, 5=Outpatient, 6=Inpatient-RefProvReq

In [ ]:
%sql
-- Check how many visits we have in the source
SELECT 
    COUNT(*) as total_visits,
    COUNT(DISTINCT ID) as unique_visit_ids,
    COUNT(DISTINCT PatientID) as unique_patients,
    MIN(StartDTTM) as earliest_visit,
    MAX(StartDTTM) as latest_visit
FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_visit`
WHERE ID IS NOT NULL 
  AND PatientID IS NOT NULL

# Transformation

In [ ]:
source = 'allscripts_tw'

In [ ]:
silver_visit_df = spark.sql(f'''
SELECT 
  source_to_person.person_id,
  COALESCE(visit_type_concept.omop_concept_id, 9202) AS visit_concept_id,  -- Default: 9202 = Outpatient Visit
  CAST(v.StartDTTM AS DATE) AS visit_start_date,
  v.StartDTTM AS visit_start_datetime,
  CAST(COALESCE(v.EndDTTM, v.StartDTTM) AS DATE) AS visit_end_date,
  COALESCE(v.EndDTTM, v.StartDTTM) AS visit_end_datetime,
  32817 AS visit_type_concept_id,  -- 32817 = EHR
  NULL AS provider_id,  -- TODO: Map v.PrimaryProviderID once provider table is populated
  NULL AS care_site_id,  -- TODO: Map v.LocationDE or v.BillingLocationDE once care_site table is populated
  CONCAT('{source}', ' | ', v.ID) AS visit_source_value,
  0 AS visit_source_concept_id,
  0 AS admitted_from_concept_id,  -- TODO: Map if admission source data is available
  NULL AS admitted_from_source_value,
  0 AS discharged_to_concept_id,  -- TODO: Map if discharge destination data is available
  NULL AS discharged_to_source_value,
  NULL AS preceding_visit_occurrence_id,  -- TODO: Add logic for linking sequential visits
  '{source}' AS source_system
FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_visit` v
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('{source}', CHAR(31), 'dbo_person', CHAR(31), 'id', CHAR(31), CAST(v.PatientID AS BIGINT)) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept visit_type_concept
  ON visit_type_concept.source_id = v.VisitTypeDE
  AND visit_type_concept.domain_id = 'Visit'
  AND visit_type_concept.source_system = '{source}'
WHERE v.ID IS NOT NULL
  AND v.PatientID IS NOT NULL
  AND v.StartDTTM IS NOT NULL
''')

display(silver_visit_df)
silver_visit_df.createOrReplaceTempView("silver_visit_occurrence")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.visit_occurrence AS t
USING (
  SELECT * FROM silver_visit_occurrence 
  WHERE visit_start_date IS NOT NULL
) AS s
ON t.visit_source_value = s.visit_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.visit_concept_id <=> s.visit_concept_id)
  OR NOT (t.visit_start_date <=> s.visit_start_date)
  OR NOT (t.visit_start_datetime <=> s.visit_start_datetime)
  OR NOT (t.visit_end_date <=> s.visit_end_date)
  OR NOT (t.visit_end_datetime <=> s.visit_end_datetime)
  OR NOT (t.visit_type_concept_id <=> s.visit_type_concept_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.care_site_id <=> s.care_site_id)
  OR NOT (t.visit_source_concept_id <=> s.visit_source_concept_id)
  OR NOT (t.admitted_from_concept_id <=> s.admitted_from_concept_id)
  OR NOT (t.admitted_from_source_value <=> s.admitted_from_source_value)
  OR NOT (t.discharged_to_concept_id <=> s.discharged_to_concept_id)
  OR NOT (t.discharged_to_source_value <=> s.discharged_to_source_value)
  OR NOT (t.preceding_visit_occurrence_id <=> s.preceding_visit_occurrence_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                      = s.person_id,
  t.visit_concept_id               = s.visit_concept_id,
  t.visit_start_date               = s.visit_start_date,
  t.visit_start_datetime           = s.visit_start_datetime,
  t.visit_end_date                 = s.visit_end_date,
  t.visit_end_datetime             = s.visit_end_datetime,
  t.visit_type_concept_id          = s.visit_type_concept_id,
  t.provider_id                    = s.provider_id,
  t.care_site_id                   = s.care_site_id,
  t.visit_source_concept_id        = s.visit_source_concept_id,
  t.admitted_from_concept_id       = s.admitted_from_concept_id,
  t.admitted_from_source_value     = s.admitted_from_source_value,
  t.discharged_to_concept_id       = s.discharged_to_concept_id,
  t.discharged_to_source_value     = s.discharged_to_source_value,
  t.preceding_visit_occurrence_id  = s.preceding_visit_occurrence_id,
  t.source_system                  = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id,
  source_system
)
VALUES (
  s.person_id,
  s.visit_concept_id,
  s.visit_start_date,
  s.visit_start_datetime,
  s.visit_end_date,
  s.visit_end_datetime,
  s.visit_type_concept_id,
  s.provider_id,
  s.care_site_id,
  s.visit_source_value,
  s.visit_source_concept_id,
  s.admitted_from_concept_id,
  s.admitted_from_source_value,
  s.discharged_to_concept_id,
  s.discharged_to_source_value,
  s.preceding_visit_occurrence_id,
  s.source_system
);

In [ ]:
%sql
SELECT * FROM _exponent.omop_silver.visit_occurrence
LIMIT 10

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_occurrence (
    source_system,
    visit_occurrence_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.visit_source_value AS visit_occurrence_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        visit_source_value,
        person_id
    FROM _exponent.omop_silver.visit_occurrence
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visit_occurrence x
  ON s.visit_source_value = x.visit_occurrence_source_value;

In [ ]:
%sql
SELECT * FROM _exponent.omop_mapping.source_to_visit_occurrence
LIMIT 20

In [ ]:
%sql
MERGE INTO _exponent.omop.visit_occurrence AS gold_visit
USING (
  SELECT
    source_to_visit_occurrence.visit_occurrence_id,
    s.person_id,
    s.visit_concept_id,
    s.visit_start_date,
    s.visit_start_datetime,
    s.visit_end_date,
    s.visit_end_datetime,
    s.visit_type_concept_id,
    s.provider_id,
    s.care_site_id,
    s.visit_source_value,
    s.visit_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    s.preceding_visit_occurrence_id
  FROM _exponent.omop_silver.visit_occurrence s
  JOIN _exponent.omop_mapping.source_to_visit_occurrence
    ON source_to_visit_occurrence.visit_occurrence_source_value = s.visit_source_value
   AND source_to_visit_occurrence.active_flag = TRUE
) AS src
ON gold_visit.visit_occurrence_id = src.visit_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold_visit.person_id                     = src.person_id,
  gold_visit.visit_concept_id              = src.visit_concept_id,
  gold_visit.visit_start_date              = src.visit_start_date,
  gold_visit.visit_start_datetime          = src.visit_start_datetime,
  gold_visit.visit_end_date                = src.visit_end_date,
  gold_visit.visit_end_datetime            = src.visit_end_datetime,
  gold_visit.visit_type_concept_id         = src.visit_type_concept_id,
  gold_visit.provider_id                   = src.provider_id,
  gold_visit.care_site_id                  = src.care_site_id,
  gold_visit.visit_source_value            = src.visit_source_value,
  gold_visit.visit_source_concept_id       = src.visit_source_concept_id,
  gold_visit.admitted_from_concept_id      = src.admitted_from_concept_id,
  gold_visit.admitted_from_source_value    = src.admitted_from_source_value,
  gold_visit.discharged_to_concept_id      = src.discharged_to_concept_id,
  gold_visit.discharged_to_source_value    = src.discharged_to_source_value,
  gold_visit.preceding_visit_occurrence_id = src.preceding_visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
  visit_occurrence_id,
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id
)
VALUES (
  src.visit_occurrence_id,
  src.person_id,
  src.visit_concept_id,
  src.visit_start_date,
  src.visit_start_datetime,
  src.visit_end_date,
  src.visit_end_datetime,
  src.visit_type_concept_id,
  src.provider_id,
  src.care_site_id,
  src.visit_source_value,
  src.visit_source_concept_id,
  src.admitted_from_concept_id,
  src.admitted_from_source_value,
  src.discharged_to_concept_id,
  src.discharged_to_source_value,
  src.preceding_visit_occurrence_id
);

In [ ]:
%sql
SELECT * FROM _exponent.omop.visit_occurrence
LIMIT 20

### Source Tables:
- _exponent._bronze_allscripts_tw_works_vw.dbo_visit (main visit table)
- _exponent._bronze_allscripts_tw_works_vw.dbo_visit_detail (admission/discharge details)
- _exponent._bronze_allscripts_tw_works_vw.dbo_visit_type_de (visit type lookup)

### To Do:
- Map VisitTypeDE to OMOP visit_concept_id using domain_source_to_concept
- Map visit_type_concept_id (32817 = EHR is default)
- Link to provider_id using PrimaryProviderID once provider table is populated
- Link to care_site_id using LocationDE once care_site table is populated
- Map admitted_from and discharged_to concepts

### Notes:
- PERSON must run before VISIT_OCCURRENCE (needs person_id mapping)
- PROVIDER and CARE_SITE should ideally run first but are optional
- dbo_visit.ID is the visit identifier (229M unique visits)
- dbo_visit.PatientID links to dbo_person.ID
- Visit types: 2=Office, 3=Telephone, 4=Inpatient, 5=Outpatient, 6=Inpatient-RefProvReq

In [ ]:
%sql
-- Check how many visits we have in the source
SELECT 
    COUNT(*) as total_visits,
    COUNT(DISTINCT ID) as unique_visit_ids,
    COUNT(DISTINCT PatientID) as unique_patients,
    MIN(StartDTTM) as earliest_visit,
    MAX(StartDTTM) as latest_visit
FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_visit`
WHERE ID IS NOT NULL 
  AND PatientID IS NOT NULL

# Transformation

In [ ]:
source = 'allscripts_tw'

In [ ]:
silver_visit_df = spark.sql(f'''
SELECT 
  source_to_person.person_id,
  COALESCE(visit_type_concept.omop_concept_id, 9202) AS visit_concept_id,  -- Default: 9202 = Outpatient Visit
  CAST(v.StartDTTM AS DATE) AS visit_start_date,
  v.StartDTTM AS visit_start_datetime,
  CAST(COALESCE(v.EndDTTM, v.StartDTTM) AS DATE) AS visit_end_date,
  COALESCE(v.EndDTTM, v.StartDTTM) AS visit_end_datetime,
  32817 AS visit_type_concept_id,  -- 32817 = EHR
  NULL AS provider_id,  -- TODO: Map v.PrimaryProviderID once provider table is populated
  NULL AS care_site_id,  -- TODO: Map v.LocationDE or v.BillingLocationDE once care_site table is populated
  CONCAT('{source}', ' | ', v.ID) AS visit_source_value,
  COALESCE(visit_type_concept.source_concept_id, 0) AS visit_source_concept_id,
  0 AS admitted_from_concept_id,  -- TODO: Map if admission source data is available
  NULL AS admitted_from_source_value,
  0 AS discharged_to_concept_id,  -- TODO: Map if discharge destination data is available
  NULL AS discharged_to_source_value,
  NULL AS preceding_visit_occurrence_id,  -- TODO: Add logic for linking sequential visits
  '{source}' AS source_system
FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_visit` v
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('{source}', ' | ', v.PatientID) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept visit_type_concept
  ON visit_type_concept.source_id = v.VisitTypeDE
  AND visit_type_concept.domain_id = 'Visit'
  AND visit_type_concept.source_system = '{source}'
WHERE v.ID IS NOT NULL
  AND v.PatientID IS NOT NULL
''')

display(silver_visit_df)
silver_visit_df.createOrReplaceTempView("silver_visit_occurrence")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.visit_occurrence AS t
USING silver_visit_occurrence AS s
ON t.visit_source_value = s.visit_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.visit_concept_id <=> s.visit_concept_id)
  OR NOT (t.visit_start_date <=> s.visit_start_date)
  OR NOT (t.visit_start_datetime <=> s.visit_start_datetime)
  OR NOT (t.visit_end_date <=> s.visit_end_date)
  OR NOT (t.visit_end_datetime <=> s.visit_end_datetime)
  OR NOT (t.visit_type_concept_id <=> s.visit_type_concept_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.care_site_id <=> s.care_site_id)
  OR NOT (t.visit_source_concept_id <=> s.visit_source_concept_id)
  OR NOT (t.admitted_from_concept_id <=> s.admitted_from_concept_id)
  OR NOT (t.admitted_from_source_value <=> s.admitted_from_source_value)
  OR NOT (t.discharged_to_concept_id <=> s.discharged_to_concept_id)
  OR NOT (t.discharged_to_source_value <=> s.discharged_to_source_value)
  OR NOT (t.preceding_visit_occurrence_id <=> s.preceding_visit_occurrence_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                      = s.person_id,
  t.visit_concept_id               = s.visit_concept_id,
  t.visit_start_date               = s.visit_start_date,
  t.visit_start_datetime           = s.visit_start_datetime,
  t.visit_end_date                 = s.visit_end_date,
  t.visit_end_datetime             = s.visit_end_datetime,
  t.visit_type_concept_id          = s.visit_type_concept_id,
  t.provider_id                    = s.provider_id,
  t.care_site_id                   = s.care_site_id,
  t.visit_source_concept_id        = s.visit_source_concept_id,
  t.admitted_from_concept_id       = s.admitted_from_concept_id,
  t.admitted_from_source_value     = s.admitted_from_source_value,
  t.discharged_to_concept_id       = s.discharged_to_concept_id,
  t.discharged_to_source_value     = s.discharged_to_source_value,
  t.preceding_visit_occurrence_id  = s.preceding_visit_occurrence_id,
  t.source_system                  = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id,
  source_system
)
VALUES (
  s.person_id,
  s.visit_concept_id,
  s.visit_start_date,
  s.visit_start_datetime,
  s.visit_end_date,
  s.visit_end_datetime,
  s.visit_type_concept_id,
  s.provider_id,
  s.care_site_id,
  s.visit_source_value,
  s.visit_source_concept_id,
  s.admitted_from_concept_id,
  s.admitted_from_source_value,
  s.discharged_to_concept_id,
  s.discharged_to_source_value,
  s.preceding_visit_occurrence_id,
  s.source_system
);

In [ ]:
%sql
SELECT * FROM _exponent.omop_silver.visit_occurrence
LIMIT 10

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visitor_occurrence (
    source_system,
    visit_occurrence_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.visit_source_value AS visit_occurrence_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        visit_source_value,
        person_id
    FROM _exponent.omop_silver.visit_occurrence
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_visitor_occurrence x
  ON s.visit_source_value = x.visit_occurrence_source_value;

In [ ]:
%sql
SELECT * FROM _exponent.omop_mapping.source_to_visitor_occurrence
LIMIT 20

In [ ]:
%sql
MERGE INTO _exponent.omop.visit_occurrence AS gold_visit
USING (
  SELECT
    source_to_visitor_occurrence.visit_occurrence_id,
    s.person_id,
    s.visit_concept_id,
    s.visit_start_date,
    s.visit_start_datetime,
    s.visit_end_date,
    s.visit_end_datetime,
    s.visit_type_concept_id,
    s.provider_id,
    s.care_site_id,
    s.visit_source_value,
    s.visit_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    s.preceding_visit_occurrence_id
  FROM _exponent.omop_silver.visit_occurrence s
  JOIN _exponent.omop_mapping.source_to_visitor_occurrence
    ON source_to_visitor_occurrence.visit_occurrence_source_value = s.visit_source_value
   AND source_to_visitor_occurrence.active_flag = TRUE
) AS src
ON gold_visit.visit_occurrence_id = src.visit_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold_visit.person_id                     = src.person_id,
  gold_visit.visit_concept_id              = src.visit_concept_id,
  gold_visit.visit_start_date              = src.visit_start_date,
  gold_visit.visit_start_datetime          = src.visit_start_datetime,
  gold_visit.visit_end_date                = src.visit_end_date,
  gold_visit.visit_end_datetime            = src.visit_end_datetime,
  gold_visit.visit_type_concept_id         = src.visit_type_concept_id,
  gold_visit.provider_id                   = src.provider_id,
  gold_visit.care_site_id                  = src.care_site_id,
  gold_visit.visit_source_value            = src.visit_source_value,
  gold_visit.visit_source_concept_id       = src.visit_source_concept_id,
  gold_visit.admitted_from_concept_id      = src.admitted_from_concept_id,
  gold_visit.admitted_from_source_value    = src.admitted_from_source_value,
  gold_visit.discharged_to_concept_id      = src.discharged_to_concept_id,
  gold_visit.discharged_to_source_value    = src.discharged_to_source_value,
  gold_visit.preceding_visit_occurrence_id = src.preceding_visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
  visit_occurrence_id,
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id
)
VALUES (
  src.visit_occurrence_id,
  src.person_id,
  src.visit_concept_id,
  src.visit_start_date,
  src.visit_start_datetime,
  src.visit_end_date,
  src.visit_end_datetime,
  src.visit_type_concept_id,
  src.provider_id,
  src.care_site_id,
  src.visit_source_value,
  src.visit_source_concept_id,
  src.admitted_from_concept_id,
  src.admitted_from_source_value,
  src.discharged_to_concept_id,
  src.discharged_to_source_value,
  src.preceding_visit_occurrence_id
);

In [ ]:
%sql
SELECT * FROM _exponent.omop.visit_occurrence
LIMIT 20